# 02. Treinar Splink (link_only Censo × CPF)

Profile, blocking pré-treino e treino do modelo. Grava `splink_model.json`.
Predict e clustering ficam no [`02b_aplicar_splink.ipynb`](02b_aplicar_splink.ipynb).

Usa `link_type='link_only'`: só gera pares **entre** Censo e CPF.
Validação: [`03_avaliar.ipynb`](03_avaliar.ipynb).
Lista operacional: [`04_atribuir.ipynb`](04_atribuir.ipynb).

**CPF da coorte:** `cpf_norm` no Censo vem da `cohort_dedup` (NB00/00b, MIN se ambíguo); no CPF já existia.
Entra no prior determinístico e num EM. **Não** entra no blocking de predição
(as 12 regras) nem no score — senão a GT sempre seria candidata e o 03
circularia.

**1ª passada:** sem nome da mãe. Nomes fonéticos com exact + JW 0,95 e 0,92.
Data: `data_nascimento` e partes (`ano_nascimento` / `mes_nascimento` /
`dia_nascimento`) com Null → Exact → Damerau ≤ 1 → Else. TF só na data
completa e no ano (mês tem 12 valores, dia 31). Idade exact e ±1. UF no score.
Sexo não entra na nota. Blocking de predição: 12 regras em `splink_spec` (nome sem sexo;
sem nome: DOB+CEP e DOB+UF+sexo).

Data completa exact implica ano+mês+dia exact — o EM ainda estima, mas os
pesos se sobrepõem um pouco, como nome completo + partes. Não usamos
`DateOfBirthComparison` (diffs de mês/ano em época).

Requer Splink 5 (`pip install 'splink==5.0.0.dev1'`).

**EDA descritiva:** [`01_analise_descritiva.ipynb`](01_analise_descritiva.ipynb).


In [ ]:
import sys
from pathlib import Path

PROB_DIR = Path.cwd()
if PROB_DIR.name == 'notebooks':
    PROB_DIR = PROB_DIR.parent
if str(PROB_DIR) not in sys.path:
    sys.path.insert(0, str(PROB_DIR))

from config import (
    DUCKDB_MEMORY_LIMIT,
    DUCKDB_THREADS,
    SPLINK_INPUT_VIEW,
    SPLINK_MODEL_JSON,
    TABELA_CENSO_LIMPA,
    TABELA_CPF_LIMPA,
    drop_splink_temp_tables,
    get_connection,
    get_splink_db_api,
    materialize_splink_input,
    print_paths,
    require_tables,
)

print_paths()
con = get_connection()
drop_splink_temp_tables(con)
require_tables(con, [TABELA_CENSO_LIMPA, TABELA_CPF_LIMPA], notebook_origem='00b')
materialize_splink_input(con)
db_api = get_splink_db_api(con)

n_reg = con.execute(f'SELECT COUNT(*) FROM {SPLINK_INPUT_VIEW}').fetchone()[0]
duck_settings = con.execute(
    "SELECT current_setting('threads'), current_setting('memory_limit')"
).fetchone()
print(f'Registros: {n_reg:,}')
print(
    f'DuckDB: threads={duck_settings[0]}, memory_limit={duck_settings[1]} '
    f'(defaults: {DUCKDB_THREADS}, {DUCKDB_MEMORY_LIMIT})'
)

SPLINK_ANALYSIS_SAMPLE_N = 1_000_000
analysis_table = SPLINK_INPUT_VIEW
if n_reg > SPLINK_ANALYSIS_SAMPLE_N:
    con.execute(f'''
    CREATE OR REPLACE TEMP TABLE splink_analysis_sample AS
    SELECT * FROM {SPLINK_INPUT_VIEW}
    USING SAMPLE {SPLINK_ANALYSIS_SAMPLE_N} ROWS
    ''')
    analysis_table = 'splink_analysis_sample'
    print(f'Amostra profile/blocking: {SPLINK_ANALYSIS_SAMPLE_N:,} de {n_reg:,}')


## Sanity check - composição da base

Volume por fonte e preenchimento das colunas usadas no linkage. Blocking em
coluna muito vazia gera poucos pares candidatos.


In [ ]:
from IPython.display import display

display(con.execute(f'''
SELECT origem, COUNT(*) AS n
FROM {SPLINK_INPUT_VIEW} GROUP BY 1 ORDER BY 2 DESC
''').df())

LINKAGE_COLS = [
    'primeiro_nome', 'ultimo_nome', 'nome_completo','nome_meio',
    'nome_mae', 'primeiro_nome_mae', 'nome_meio_mae', 'ultimo_nome_mae',
    'data_nascimento', 'ano_nascimento', 'mes_nascimento', 'dia_nascimento',
    'idade', 'cep', 'sexo', 'uf',
    'nome_completo_phon','primeiro_nome_phon','nome_meio_phon','ultimo_nome_phon',
    'nome_mae_phon','primeiro_nome_mae_phon','nome_meio_mae_phon','ultimo_nome_mae_phon'
]

cols_presentes = [
    c for c in LINKAGE_COLS
    if c in set(con.execute(f'SELECT * FROM {SPLINK_INPUT_VIEW} LIMIT 0').df().columns)
]
preenchimento = ',\n    '.join(
    f"ROUND(100.0 * COUNT({c}) / COUNT(*), 1) AS pct_{c}" for c in cols_presentes
)
display(con.execute(f'''
SELECT origem, {preenchimento}
FROM {SPLINK_INPUT_VIEW} GROUP BY origem ORDER BY origem
''').df().T)


## Exploração pré-modelo

Profile Splink das colunas de linkage e análise de blocking (cumulativo + maiores blocos).


In [ ]:
from splink import block_on
from splink.blocking_analysis import cumulative_comparisons_to_be_scored_from_blocking_rules_chart
from splink.exploratory import profile_columns
from splink_spec import BLOCKING_RULE_COLUMNS

blocking_rules = [block_on(*cols) for cols in BLOCKING_RULE_COLUMNS]

profile_columns(
    con.execute(f'SELECT * FROM {analysis_table}').df(),
    db_api,
    column_expressions=[
        'primeiro_nome_phon', 'ultimo_nome_phon', 'nome_completo_phon',
        'uf', 'data_nascimento', 'ano_nascimento', 'mes_nascimento', 'dia_nascimento',
        'idade', 'cpf_norm',
    ],
)


In [ ]:
# Análise de blocking só entre origens (link_only), na amostra ou na base completa.
con.execute(f'''
CREATE OR REPLACE VIEW splink_analysis_censo AS
SELECT * FROM {analysis_table} WHERE origem = 'censo'
''')
con.execute(f'''
CREATE OR REPLACE VIEW splink_analysis_cpf AS
SELECT * FROM {analysis_table} WHERE origem = 'cpf'
''')

cumulative_comparisons_to_be_scored_from_blocking_rules_chart(
    table_or_tables=['splink_analysis_censo', 'splink_analysis_cpf'],
    blocking_rules=blocking_rules,
    db_api=db_api,
    link_type='link_only',
)


## Modelo Splink

Settings e `Linker` em `link_only`. Comparisons: nomes fonéticos (JW 0,95 e 0,92),
data completa e partes (Exact + Damerau ≤ 1), idade (exact e ±1), UF.
Sem `nome_mae*`, sem sexo, sem CEP e **sem CPF** no score. UF no score.
CEP entra numa regra de predição sem nome (DOB+CEP) e no EM.
Sexo entra só na regra sem nome DOB+UF+sexo, não nas regras com nome.
`cpf_norm` entra no prior (`deterministic_rules`) e num EM; `NULL = NULL` é
falso, então Censo sem ouro não fabrica par.


In [ ]:
from splink import Linker, SettingsCreator
import splink.comparison_library as cl
import splink.comparison_level_library as cll


def dob_comparison(col, *, term_frequency_adjustments=False):
    return cl.CustomComparison(
        comparison_levels=[
            cll.NullLevel(col),
            cll.ExactMatchLevel(col, term_frequency_adjustments=term_frequency_adjustments),
            cll.DamerauLevenshteinLevel(col, 1),
            cll.ElseLevel(),
        ],
    )


comparisons = [
    cl.NameComparison('nome_completo_phon', jaro_winkler_thresholds=[0.95, 0.92]).configure(term_frequency_adjustments=True),
    cl.NameComparison('primeiro_nome_phon', jaro_winkler_thresholds=[0.95, 0.92]).configure(term_frequency_adjustments=True),
    cl.NameComparison('nome_meio_phon', jaro_winkler_thresholds=[0.95, 0.92]).configure(term_frequency_adjustments=True),
    cl.NameComparison('ultimo_nome_phon', jaro_winkler_thresholds=[0.95, 0.92]).configure(term_frequency_adjustments=True),
    dob_comparison('data_nascimento', term_frequency_adjustments=True),
    dob_comparison('ano_nascimento', term_frequency_adjustments=True),
    dob_comparison('mes_nascimento'),
    dob_comparison('dia_nascimento'),
    cl.CustomComparison(
        comparison_levels=[
            cll.NullLevel('idade'),
            cll.ExactMatchLevel('idade'),
            cll.AbsoluteDifferenceLevel('idade', 1),
            cll.ElseLevel(),
        ],
    ),
    cl.ExactMatch('uf').configure(term_frequency_adjustments=True),
]

con.execute(f'''
CREATE OR REPLACE VIEW splink_censo AS
SELECT * FROM {SPLINK_INPUT_VIEW} WHERE origem = 'censo'
''')
con.execute(f'''
CREATE OR REPLACE VIEW splink_cpf AS
SELECT * FROM {SPLINK_INPUT_VIEW} WHERE origem = 'cpf'
''')
print(
    'splink_censo:', con.execute('SELECT COUNT(*) FROM splink_censo').fetchone()[0],
    '| splink_cpf:', con.execute('SELECT COUNT(*) FROM splink_cpf').fetchone()[0],
)

settings = SettingsCreator(
    link_type='link_only',
    unique_id_column_name='unique_id',
    comparisons=comparisons,
    blocking_rules_to_generate_predictions=blocking_rules,
    retain_intermediate_calculation_columns=True,
)
linker = Linker(
    ['splink_censo', 'splink_cpf'],
    settings,
    db_api=db_api,
    input_table_aliases=['censo', 'cpf'],
)


In [ ]:
deterministic_rules = [
    block_on('nome_completo_phon', 'data_nascimento'),
    block_on('cpf_norm'),
]

linker.training.estimate_probability_two_random_records_match(deterministic_rules, recall=0.7)
linker.training.estimate_u_using_random_sampling(max_pairs=10_000_000)
# EM em sexo+DOB (sem CEP): observa discordância de nome.
linker.training.estimate_parameters_using_expectation_maximisation(
    block_on('sexo', 'data_nascimento'),
    estimate_without_term_frequencies=True,
)


In [ ]:
# EM em primeiro_nome_phon + DOB: observa m de sobrenome, completo, idade.
linker.training.estimate_parameters_using_expectation_maximisation(
    block_on('primeiro_nome_phon', 'data_nascimento'),
    estimate_without_term_frequencies=True,
)


In [ ]:
# EM em nome + sexo + UF + CEP: observa m de data/idade.
linker.training.estimate_parameters_using_expectation_maximisation(
    block_on('primeiro_nome_phon', 'ultimo_nome_phon','sexo','uf','cep'),
    estimate_without_term_frequencies=True,
)


In [ ]:
# EM em CPF ouro 1:1: bloco quase só match (u permanece o do random sampling).
# Fora do blocking_rules_to_generate_predictions — o JSON de predição não leva esta regra.
linker.training.estimate_parameters_using_expectation_maximisation(
    block_on('cpf_norm'),
    estimate_without_term_frequencies=True,
)


In [ ]:
from config import MODELS_DIR

SPLINK_MODEL_JSON.parent.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)
linker.misc.save_model_to_json(str(SPLINK_MODEL_JSON), overwrite=True)
linker.misc.save_model_to_json(str(MODELS_DIR / 'splink_model.json'), overwrite=True)
print('Modelo salvo:', SPLINK_MODEL_JSON)
print('Cópia em models/:', MODELS_DIR / 'splink_model.json')


## Visualização pós-treino

Match weights e registros difíceis de linkar (unlinkables).


In [ ]:
linker.visualisations.match_weights_chart()


In [ ]:
linker.evaluation.unlinkables_chart()


## Encerrar

JSON pronto para o [`02b_aplicar_splink.ipynb`](02b_aplicar_splink.ipynb).


In [ ]:
print(f"{'modelo':12s} {'ok ' if SPLINK_MODEL_JSON.exists() else 'FALTA'} {SPLINK_MODEL_JSON}")


In [ ]:
con.close()
